<a href="https://colab.research.google.com/github/Udaythecoder/BLUESTOCK/blob/main/BDay_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Cleaning: `nav_history.csv`

This section focuses on cleaning the `nav_history.csv` dataset. The steps include:

1.  **Loading the Data**: Reading the CSV into a pandas DataFrame.
2.  **Date Parsing**: Converting the 'date' column to datetime objects.
3.  **Sorting**: Ordering the data by 'amfi_code' and 'date'.
4.  **Forward-Filling NAV**: Handling missing NAV values, particularly for holidays/weekends.
5.  **Removing Duplicates**: Ensuring unique entries.
6.  **Validating NAV**: Checking that all 'NAV' values are positive.

In [1]:
import pandas as pd

# Load the nav_history.csv file
nav_df = pd.read_csv('/content/02_nav_history.csv')

print("Original DataFrame info:")
nav_df.info()
print("\nFirst 5 rows of original DataFrame:")
display(nav_df.head())

Original DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB

First 5 rows of original DataFrame:


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


### Step 1: Parse 'date' column to datetime objects

Converting the 'date' column to datetime objects allows for proper time-series operations and sorting.

In [2]:
nav_df['date'] = pd.to_datetime(nav_df['date'])

print("DataFrame info after date parsing:")
nav_df.info()

DataFrame info after date parsing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


### Step 2: Sort by 'amfi_code' and 'date'

Sorting ensures that the data for each `amfi_code` is in chronological order, which is crucial for operations like forward-filling.

In [3]:
nav_df = nav_df.sort_values(by=['amfi_code', 'date'])

print("First 5 rows after sorting:")
display(nav_df.head())

First 5 rows after sorting:


,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


### Step 3: Forward-fill missing 'NAV' values per `amfi_code`

Missing NAV values (e.g., for holidays or weekends) are filled using the last valid observation for each `amfi_code`. This ensures continuity in the time series data.

In [5]:
nav_df['nav'] = nav_df.groupby('amfi_code')['nav'].ffill()

print("Missing NAV values after forward-fill:")
print(nav_df['nav'].isnull().sum())

Missing NAV values after forward-fill:
0


### Step 4: Remove duplicate rows

Ensuring that there are no duplicate entries in the dataset, especially duplicates based on 'amfi_code' and 'date' combination.

In [6]:
initial_rows = len(nav_df)
nav_df.drop_duplicates(inplace=True)
print(f"Number of rows before removing duplicates: {initial_rows}")
print(f"Number of rows after removing duplicates: {len(nav_df)}")

Number of rows before removing duplicates: 46000
Number of rows after removing duplicates: 46000


### Step 5: Validate 'NAV' values are greater than 0

This step checks for any non-positive NAV values, which might indicate data entry errors or anomalies.

In [8]:
invalid_nav_count = nav_df[nav_df['nav'] <= 0].shape[0]
print(f"Number of NAV values less than or equal to 0: {invalid_nav_count}")

if invalid_nav_count > 0:
    print("Rows with invalid NAV values:")
    display(nav_df[nav_df['NAV'] <= 0])
    # Optionally, you might want to drop these rows or handle them differently
    # nav_df = nav_df[nav_df['NAV'] > 0]

print("\nCleaning complete. Final DataFrame info:")
nav_df.info()
print("\nFirst 5 rows of cleaned DataFrame:")
display(nav_df.head())

Number of NAV values less than or equal to 0: 0

Cleaning complete. Final DataFrame info:
<class 'pandas.core.frame.DataFrame'>
Index: 46000 entries, 5750 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.4 MB

First 5 rows of cleaned DataFrame:


,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


## Data Cleaning: `investor_transactions.csv`

This section focuses on cleaning the `investor_transactions.csv` dataset. The steps include:

1.  **Loading the Data**: Reading the CSV into a pandas DataFrame.
2.  **Standardize Transaction Type**: Mapping various transaction type strings to 'SIP', 'Lumpsum', or 'Redemption'.
3.  **Validate Amount**: Ensuring that all transaction amounts are positive.
4.  **Fix Date Formats**: Converting relevant columns to datetime objects.
5.  **Check KYC Status**: Validating the values in the 'KYC_status' column.

In [9]:
import pandas as pd

# Load the investor_transactions.csv file
transactions_df = pd.read_csv('/content/08_investor_transactions.csv')

print("Original DataFrame info:")
transactions_df.info()
print("\nFirst 5 rows of original DataFrame:")
display(transactions_df.head())

Original DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB

First 5 rows of original DataFrame:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


### Step 1: Standardize 'transaction_type' values

We will map various input strings to a consistent set of values: 'SIP', 'Lumpsum', or 'Redemption'. Any values that don't fit these categories will be flagged or handled as 'Other'.

In [10]:
transaction_type_mapping = {
    'SIP': 'SIP',
    'Systematic Investment Plan': 'SIP',
    'Lumpsum': 'Lumpsum',
    'One-time Investment': 'Lumpsum',
    'Redemption': 'Redemption',
    'Withdrawal': 'Redemption'
}

transactions_df['transaction_type'] = transactions_df['transaction_type'].replace(transaction_type_mapping)

# Check for any unmapped transaction types
unmapped_types = transactions_df[~transactions_df['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption'])]['transaction_type'].unique()
if len(unmapped_types) > 0:
    print(f"Found unmapped transaction types: {unmapped_types}. Consider adding them to the mapping.")

print("Transaction type value counts after standardization:")
display(transactions_df['transaction_type'].value_counts())

Transaction type value counts after standardization:


,count
transaction_type,
SIP,19716
Lumpsum,8095
Redemption,4967


### Step 2: Validate 'amount' > 0

Ensuring that all transaction amounts are positive, as negative or zero amounts would be invalid for these transaction types.

In [12]:
invalid_amount_count = transactions_df[transactions_df['amount_inr'] <= 0].shape[0]
print(f"Number of transactions with amount <= 0: {invalid_amount_count}")

if invalid_amount_count > 0:
    print("Rows with invalid amounts:")
    display(transactions_df[transactions_df['amount_inr'] <= 0])
    # Optionally, remove or correct these rows
    # transactions_df = transactions_df[transactions_df['amount_inr'] > 0]

Number of transactions with amount <= 0: 0


### Step 3: Fix 'transaction_date' formats

Converting the 'transaction_date' column to datetime objects for accurate time-series analysis.

In [13]:
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'], errors='coerce')

missing_dates_count = transactions_df['transaction_date'].isnull().sum()
print(f"Number of invalid dates after conversion: {missing_dates_count}")

if missing_dates_count > 0:
    print("Rows with invalid transaction dates (original):")
    display(transactions_df[transactions_df['transaction_date'].isnull()])
    # Optionally, remove these rows or investigate the original format
    # transactions_df.dropna(subset=['transaction_date'], inplace=True)

print("DataFrame info after date conversion:")
transactions_df.info()

Number of invalid dates after conversion: 0
DataFrame info after date conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          

### Step 4: Check 'KYC_status' enum values

Validating that the 'KYC_status' column contains only expected categorical values (e.g., 'Completed', 'Pending', 'Rejected').

In [15]:
expected_kyc_statuses = ['Completed', 'Pending', 'Rejected', 'Exempt'] # Add other valid statuses if applicable

unexpected_kyc_statuses = transactions_df[~transactions_df['kyc_status'].isin(expected_kyc_statuses)]['kyc_status'].unique()

if len(unexpected_kyc_statuses) > 0:
    print(f"Found unexpected KYC_status values: {unexpected_kyc_statuses}")
    # Optionally, you might want to map these to a standard value or flag them
else:
    print("All KYC_status values are as expected.")

print("KYC_status value counts:")
display(transactions_df['kyc_status'].value_counts())

Found unexpected KYC_status values: ['Verified']
KYC_status value counts:


,count
kyc_status,
Verified,30146
Pending,2632


### Cleaning Summary

The `investor_transactions.csv` dataset has been cleaned based on the specified criteria. Here's a summary of the cleaned DataFrame:

In [16]:
print("Cleaning complete. Final DataFrame info:")
transactions_df.info()
print("\nFirst 5 rows of cleaned DataFrame:")
display(transactions_df.head())

Cleaning complete. Final DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: da

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


## Data Cleaning: `scheme_performance.csv`

This section focuses on cleaning the `scheme_performance.csv` dataset. The steps include:

1.  **Loading the Data**: Reading the CSV into a pandas DataFrame.
2.  **Validate Return Values**: Ensuring all performance return columns ('1_Year_Return', '3_Year_Return', '5_Year_Return', 'Since_Inception_Return') are numeric and flagging non-numeric entries.
3.  **Flag Anomalies in Returns**: Identifying potential outliers or unusual values in the return data.
4.  **Check Expense Ratio Range**: Validating that the 'expense_ratio' is within the typical range of 0.1% to 2.5%.

In [17]:
import pandas as pd

# Load the scheme_performance.csv file
scheme_df = pd.read_csv('/content/07_scheme_performance.csv')

print("Original DataFrame info:")
scheme_df.info()
print("\nFirst 5 rows of original DataFrame:")
display(scheme_df.head())

Original DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null   

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


### Step 1: Validate all return values are numeric and flag anomalies

We will convert the return columns to numeric types. Any values that cannot be converted will become `NaN` and can be inspected as anomalies. We'll also flag return values that are unusually high or low (e.g., outside -100% to 1000% range for typical mutual fund returns).

In [19]:
return_columns = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']

for col in return_columns:
    # Convert to numeric, coercing errors to NaN
    scheme_df[col] = pd.to_numeric(scheme_df[col], errors='coerce')
    # Flag non-numeric entries (now NaN)
    non_numeric_count = scheme_df[col].isnull().sum()
    if non_numeric_count > 0:
        print(f"Found {non_numeric_count} non-numeric values in '{col}'. Displaying rows:")
        display(scheme_df[scheme_df[col].isnull()].head())
        # Optionally, fill NaNs with a suitable value like 0 or the mean/median
        # scheme_df[col].fillna(0, inplace=True)
    # Flag anomalies (e.g., returns outside a reasonable range like -100% to 1000%)
    # Assuming returns are percentages, so -1 to 10 for decimal representation
    anomalous_returns_count = scheme_df[(scheme_df[col] < -1) | (scheme_df[col] > 10)].shape[0]
    if anomalous_returns_count > 0:
        print(f"Found {anomalous_returns_count} anomalous return values in '{col}' (outside -100% to 1000%). Displaying rows:")
        display(scheme_df[(scheme_df[col] < -1) | (scheme_df[col] > 10)].head())

print("\nData types after validating return columns:")
scheme_df[return_columns].info()

Found 34 anomalous return values in 'return_1yr_pct' (outside -100% to 1000%). Displaying rows:


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate


Found 34 anomalous return values in 'return_3yr_pct' (outside -100% to 1000%). Displaying rows:


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate


Found 34 anomalous return values in 'return_5yr_pct' (outside -100% to 1000%). Displaying rows:


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate



Data types after validating return columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   return_1yr_pct  40 non-null     float64
 1   return_3yr_pct  40 non-null     float64
 2   return_5yr_pct  40 non-null     float64
dtypes: float64(3)
memory usage: 1.1 KB


### Step 2: Check 'expense_ratio' range (0.1% – 2.5%)

We will ensure the 'expense_ratio' column is numeric and then validate that its values fall within the expected range of 0.001 to 0.025 (representing 0.1% to 2.5%).

In [22]:
# Convert expense_ratio to numeric
scheme_df['expense_ratio_pct'] = pd.to_numeric(scheme_df['expense_ratio_pct'], errors='coerce')

# Flag non-numeric expense ratios
non_numeric_er_count = scheme_df['expense_ratio_pct'].isnull().sum()
if non_numeric_er_count > 0:
    print(f"Found {non_numeric_er_count} non-numeric values in 'expense_ratio_pct'. Displaying rows:")
    display(scheme_df[scheme_df['expense_ratio_pct'].isnull()].head())

# Define the valid range (0.1% to 2.5%)
min_er = 0.001  # 0.1%
max_er = 0.025  # 2.5%

# Flag expense ratios outside the valid range
invalid_er_count = scheme_df[(scheme_df['expense_ratio_pct'] < min_er) | (scheme_df['expense_ratio_pct'] > max_er)].shape[0]

if invalid_er_count > 0:
    print(f"Found {invalid_er_count} 'expense_ratio_pct' values outside the {min_er*100:.1f}% - {max_er*100:.1f}% range. Displaying rows:")
    display(scheme_df[(scheme_df['expense_ratio_pct'] < min_er) | (scheme_df['expense_ratio_pct'] > max_er)].head())
else:
    print(f"All 'expense_ratio_pct' values are within the {min_er*100:.1f}% - {max_er*100:.1f}% range.")

print("\nDataFrame info after expense ratio validation:")
scheme_df.info()

Found 40 'expense_ratio_pct' values outside the 0.1% - 2.5% range. Displaying rows:


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low



DataFrame info after expense ratio validation:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore  

### Cleaning Summary

The `scheme_performance.csv` dataset has been cleaned based on the specified criteria. Here's a summary of the cleaned DataFrame:

In [21]:
print("Cleaning complete. Final DataFrame info:")
scheme_df.info()
print("\nFirst 5 rows of cleaned DataFrame:")
display(scheme_df.head())

Cleaning complete. Final DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore         

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [59]:
!pip install sqlalchemy
!pip install ipython-sql
%load_ext sql
from sqlalchemy import create_engine
engine = create_engine('sqlite:///:memory:')
%sql sqlite:///:memory:

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Design SQLite Star Schema

We will design a star schema for our data, which typically consists of a central fact table(s) and a set of surrounding dimension tables. This structure is optimized for querying and analysis.

### Schema Overview:

*   **Dimension Tables:**
    *   `dim_fund`: Contains unique information about each mutual fund (e.g., AMFI code, scheme name, fund house).
    *   `dim_date`: Contains detailed information for each unique date present in the datasets (e.g., year, month, day, day of week).
*   **Fact Tables:**
    *   `fact_nav`: Stores daily NAV values, linking to `dim_fund` and `dim_date`.
    *   `fact_transactions`: Records investor transactions, linking to `dim_fund` and `dim_date`.
    *   `fact_performance`: Stores performance metrics for each fund, linking to `dim_fund`.
    *   `fact_aum`: Stores Assets Under Management for each fund, linking to `dim_fund`.

In [61]:
import sqlite3
from sqlalchemy import text # Import text for raw SQL execution
# --- CREATE TABLE  ---

create_dim_fund_table = '''
CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY, /* Unique identifier for each fund */
    scheme_name TEXT NOT NULL,
    fund_house TEXT NOT NULL,
    category TEXT NOT NULL,
    plan TEXT NOT NULL,
    risk_grade TEXT
);
'''

create_dim_date_table = '''
CREATE TABLE dim_date (
    date_id TEXT PRIMARY KEY, /* YYYY-MM-DD format */
    full_date DATE NOT NULL,
    day INTEGER NOT NULL,
    month INTEGER NOT NULL,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    day_of_week INTEGER NOT NULL,
    day_name TEXT NOT NULL,
    is_weekend BOOLEAN NOT NULL
);
'''

create_fact_nav_table = '''
CREATE TABLE fact_nav (
    nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL,
    date_id TEXT NOT NULL,
    nav REAL NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
);
'''

create_fact_transactions_table = '''
CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT NOT NULL,
    date_id TEXT NOT NULL, /* Link to dim_date */
    amfi_code INTEGER NOT NULL,
    transaction_type TEXT NOT NULL,
    amount_inr INTEGER NOT NULL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
);
'''

create_fact_performance_table = '''
CREATE TABLE fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
'''

create_fact_aum_table = '''
CREATE TABLE fact_aum (
    aum_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL,
    aum_crore INTEGER NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
'''

# Execute CREATE TABLE statements using the SQLAlchemy engine
with engine.connect() as connection:
    connection.execute(text(create_dim_fund_table))
    connection.execute(text(create_dim_date_table))
    connection.execute(text(create_fact_nav_table))
    connection.execute(text(create_fact_transactions_table))
    connection.execute(text(create_fact_performance_table))
    connection.execute(text(create_fact_aum_table))
    connection.commit()

print("SQLite star schema created successfully in an in-memory database.")

SQLite star schema created successfully in an in-memory database.


## 2. Load Cleaned Datasets into SQLite

Now we will load the cleaned `nav_df`, `transactions_df`, and `scheme_df` into the newly created SQLite tables. We will use `df.to_sql()` for this, ensuring that data types are consistent and foreign key constraints are respected.

In [62]:
# Create a SQLAlchemy engine for the in-memory SQLite database
# engine = create_engine('sqlite:///:memory:') # Removed to ensure it uses the existing engine

# --- Prepare and Load dim_fund ---
# Select relevant columns and drop duplicates to create the dim_fund table
df_dim_fund = scheme_df[['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'risk_grade']].drop_duplicates(subset=['amfi_code'])
df_dim_fund.to_sql('dim_fund', engine, if_exists='append', index=False)
print(f"dim_fund loaded. Rows: {len(df_dim_fund)}")

# --- Prepare and Load dim_date ---
# Extract all unique dates from nav_df and transactions_df
all_dates = pd.concat([
    nav_df['date'],
    transactions_df['transaction_date']
]).drop_duplicates().sort_values()

df_dim_date = pd.DataFrame({'full_date': all_dates})
df_dim_date['date_id'] = df_dim_date['full_date'].dt.strftime('%Y-%m-%d')
df_dim_date['day'] = df_dim_date['full_date'].dt.day
df_dim_date['month'] = df_dim_date['full_date'].dt.month
df_dim_date['year'] = df_dim_date['full_date'].dt.year
df_dim_date['quarter'] = df_dim_date['full_date'].dt.quarter
df_dim_date['day_of_week'] = df_dim_date['full_date'].dt.dayofweek
df_dim_date['day_name'] = df_dim_date['full_date'].dt.day_name()
df_dim_date['is_weekend'] = df_dim_date['full_date'].dt.dayofweek.isin([5, 6])

df_dim_date.to_sql('dim_date', engine, if_exists='append', index=False)
print(f"dim_date loaded. Rows: {len(df_dim_date)}")

# --- Prepare and Load fact_nav ---
df_fact_nav = nav_df.copy()
df_fact_nav['date_id'] = df_fact_nav['date'].dt.strftime('%Y-%m-%d')
df_fact_nav = df_fact_nav[['amfi_code', 'date_id', 'nav']]
df_fact_nav.to_sql('fact_nav', engine, if_exists='append', index=False)
print(f"fact_nav loaded. Rows: {len(df_fact_nav)}")

# --- Prepare and Load fact_transactions ---
df_fact_transactions = transactions_df.copy()
df_fact_transactions['date_id'] = df_fact_transactions['transaction_date'].dt.strftime('%Y-%m-%d')
df_fact_transactions = df_fact_transactions[[
    'investor_id', 'date_id', 'amfi_code', 'transaction_type', 'amount_inr',
    'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh',
    'payment_mode', 'kyc_status' # transaction_id is an AUTOINCREMENT primary key, not directly loaded from df
]]
df_fact_transactions.to_sql('fact_transactions', engine, if_exists='append', index=False)
print(f"fact_transactions loaded. Rows: {len(df_fact_transactions)}")

# --- Prepare and Load fact_performance ---
df_fact_performance = scheme_df[[
    'amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
    'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
    'std_dev_ann_pct', 'max_drawdown_pct', 'expense_ratio_pct', 'morningstar_rating'
]].copy()
df_fact_performance.to_sql('fact_performance', engine, if_exists='append', index=False)
print(f"fact_performance loaded. Rows: {len(df_fact_performance)}")

# --- Prepare and Load fact_aum ---
df_fact_aum = scheme_df[['amfi_code', 'aum_crore']].copy()
df_fact_aum.to_sql('fact_aum', engine, if_exists='append', index=False)
print(f"fact_aum loaded. Rows: {len(df_fact_aum)}")

print("\n--- Verification: Row Counts ---")
print(f"Source nav_df rows: {len(nav_df)}")
print(f"Fact nav rows in DB: {pd.read_sql_query('SELECT COUNT(*) FROM fact_nav', engine).iloc[0, 0]}")

print(f"Source transactions_df rows: {len(transactions_df)}")
print(f"Fact transactions rows in DB: {pd.read_sql_query('SELECT COUNT(*) FROM fact_transactions', engine).iloc[0, 0]}")

print(f"Source scheme_df rows: {len(scheme_df)}")
print(f"Fact performance rows in DB: {pd.read_sql_query('SELECT COUNT(*) FROM fact_performance', engine).iloc[0, 0]}")
print(f"Fact AUM rows in DB: {pd.read_sql_query('SELECT COUNT(*) FROM fact_aum', engine).iloc[0, 0]}")

dim_fund loaded. Rows: 40
dim_date loaded. Rows: 1296
fact_nav loaded. Rows: 46000
fact_transactions loaded. Rows: 32778
fact_performance loaded. Rows: 40
fact_aum loaded. Rows: 40

--- Verification: Row Counts ---
Source nav_df rows: 46000
Fact nav rows in DB: 46000
Source transactions_df rows: 32778
Fact transactions rows in DB: 32778
Source scheme_df rows: 40
Fact performance rows in DB: 40
Fact AUM rows in DB: 40


## 3. Analytical SQL Queries

Now that the data is loaded into the SQLite star schema, we can execute various analytical queries to gain insights. Here are 10 queries, including the ones requested:

1.  Top 5 funds by AUM
2.  Average NAV per month for a specific fund
3.  SIP Year-over-Year growth (total amount)
4.  Total transaction amount by state
5.  Funds with expense ratio < 1%
6.  Top 10 funds by 1-year return
7.  Total amount invested per investor
8.  Monthly transaction count trend
9.  Average expense ratio by fund category
10. Funds with Morningstar rating of 5 and positive 3-year return

In [63]:
print("--- SQL Query 1: Top 5 funds by AUM ---")
query1 = '''
SELECT
    df.scheme_name,
    df.fund_house,
    fa.aum_crore
FROM
    fact_aum fa
JOIN
    dim_fund df ON fa.amfi_code = df.amfi_code
ORDER BY
    fa.aum_crore DESC
LIMIT 5;
'''
display(pd.read_sql_query(query1, engine))



--- SQL Query 1: Top 5 funds by AUM ---


,scheme_name,fund_house,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,49046
1,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,47469
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,43630
3,DSP Top 100 Equity Fund - Regular - Growth,DSP Mutual Fund,41828
4,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,41728


In [64]:
print("\n--- SQL Query 2: Average NAV per month for a specific fund (e.g., AMFI Code 119551) ---")
query2 = '''
SELECT
    strftime('%Y-%m', dd.full_date) AS year_month,
    AVG(fn.nav) AS average_nav
FROM
    fact_nav fn
JOIN
    dim_date dd ON fn.date_id = dd.date_id
WHERE
    fn.amfi_code = 119551 /* Example AMFI Code */
GROUP BY
    year_month
ORDER BY
    year_month;
'''
display(pd.read_sql_query(query2, engine))


--- SQL Query 2: Average NAV per month for a specific fund (e.g., AMFI Code 119551) ---


,year_month,average_nav
0,2022-01,55.088433
1,2022-02,52.481785
2,2022-03,50.835930
3,2022-04,52.036343
4,2022-05,51.888295
5,2022-06,52.681432
6,2022-07,52.693690
7,2022-08,53.773361
8,2022-09,56.545768
9,2022-10,57.883652


In [65]:
print("\n--- SQL Query 3: SIP Year-over-Year growth (total amount) ---")
query3 = '''
WITH MonthlySIP AS (
    SELECT
        dd.year AS transaction_year,
        SUM(ft.amount_inr) AS total_sip_amount
    FROM
        fact_transactions ft
    JOIN
        dim_date dd ON ft.date_id = dd.date_id
    WHERE
        ft.transaction_type = 'SIP'
    GROUP BY
        dd.year
)
SELECT
    ms1.transaction_year,
    ms1.total_sip_amount,
    LAG(ms1.total_sip_amount, 1, 0) OVER (ORDER BY ms1.transaction_year) AS previous_year_sip_amount,
    -- Fixed: Added NULLIF to prevent division by zero for the first year's YOY growth
    (ms1.total_sip_amount - LAG(ms1.total_sip_amount, 1, 0) OVER (ORDER BY ms1.transaction_year)) * 100.0 / NULLIF(LAG(ms1.total_sip_amount, 1, 0) OVER (ORDER BY ms1.transaction_year), 0) AS yoy_growth_pct
FROM
    MonthlySIP ms1;
'''
display(pd.read_sql_query(query3, engine))


--- SQL Query 3: SIP Year-over-Year growth (total amount) ---


,transaction_year,total_sip_amount,previous_year_sip_amount,yoy_growth_pct
0,2024,153233052,0,NaN
1,2025,64000439,153233052,-58.233267


In [66]:
print("\n--- SQL Query 4: Total transaction amount by state ---")
query4 = '''
SELECT
    state,
    SUM(amount_inr) AS total_transaction_amount
FROM
    fact_transactions
GROUP BY
    state
ORDER BY
    total_transaction_amount DESC
LIMIT 10;
'''
display(pd.read_sql_query(query4, engine))


--- SQL Query 4: Total transaction amount by state ---


,state,total_transaction_amount
0,Punjab,315780459
1,Tamil Nadu,315177237
2,Madhya Pradesh,308312493
3,Rajasthan,298645822
4,Gujarat,298358940
5,West Bengal,297182514
6,Telangana,290219284
7,Delhi,289633404
8,Uttar Pradesh,285368873
9,Haryana,279634354


In [67]:
print("\n--- SQL Query 5: Funds with expense_ratio < 1% ---")
query5 = '''
SELECT
    df.scheme_name,
    df.fund_house,
    fp.expense_ratio_pct * 100 AS expense_ratio_percent
FROM
    fact_performance fp
JOIN
    dim_fund df ON fp.amfi_code = df.amfi_code
WHERE
    fp.expense_ratio_pct < 0.01 /* 1% expressed as a decimal */
ORDER BY
    fp.expense_ratio_pct ASC;
'''
display(pd.read_sql_query(query5, engine))


--- SQL Query 5: Funds with expense_ratio < 1% ---


,scheme_name,fund_house,expense_ratio_percent


In [68]:
print("\n--- SQL Query 6: Top 10 funds by 1-year return ---")
query6 = '''
SELECT
    df.scheme_name,
    df.fund_house,
    fp.return_1yr_pct * 100 AS "1_Year_Return_Percent"
FROM
    fact_performance fp
JOIN
    dim_fund df ON fp.amfi_code = df.amfi_code
WHERE
    fp.return_1yr_pct IS NOT NULL
ORDER BY
    fp.return_1yr_pct DESC
LIMIT 10;
'''
display(pd.read_sql_query(query6, engine))


--- SQL Query 6: Top 10 funds by 1-year return ---


,scheme_name,fund_house,1_Year_Return_Percent
0,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,2493.0
1,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,2456.0
2,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,2197.0
3,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,2130.0
4,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,2059.0
5,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,2020.0
6,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,1998.0
7,UTI Flexi Cap Fund - Regular - Growth,UTI Mutual Fund,1743.0
8,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,1712.0
9,ICICI Pru Value Discovery Fund - Regular - Growth,ICICI Prudential MF,1667.0


In [69]:
print("\n--- SQL Query 7: Total amount invested per investor ---")
query7 = '''
SELECT
    investor_id,
    SUM(amount_inr) AS total_invested_amount
FROM
    fact_transactions
WHERE
    transaction_type IN ('SIP', 'Lumpsum')
GROUP BY
    investor_id
ORDER BY
    total_invested_amount DESC
LIMIT 10;
'''
display(pd.read_sql_query(query7, engine))


--- SQL Query 7: Total amount invested per investor ---


,investor_id,total_invested_amount
0,INV001771,2695710
1,INV000604,2567253
2,INV001528,2546199
3,INV002707,2533904
4,INV002250,2465102
5,INV000988,2330846
6,INV002039,2265918
7,INV002996,2172258
8,INV003288,2169468
9,INV000481,2143420


In [70]:
print("\n--- SQL Query 8: Monthly transaction count trend ---")
query8 = '''
SELECT
    strftime('%Y-%m', dd.full_date) AS year_month,
    COUNT(ft.transaction_id) AS transaction_count
FROM
    fact_transactions ft
JOIN
    dim_date dd ON ft.date_id = dd.date_id
GROUP BY
    year_month
ORDER BY
    year_month;
'''
display(pd.read_sql_query(query8, engine))


--- SQL Query 8: Monthly transaction count trend ---


,year_month,transaction_count
0,2024-01,1949
1,2024-02,1861
2,2024-03,1974
3,2024-04,1952
4,2024-05,1901
5,2024-06,1989
6,2024-07,1968
7,2024-08,1985
8,2024-09,1833
9,2024-10,1957


In [71]:
print("\n--- SQL Query 9: Average expense ratio by fund category ---")
query9 = '''
SELECT
    df.category,
    AVG(fp.expense_ratio_pct) * 100 AS average_expense_ratio_percent
FROM
    fact_performance fp
JOIN
    dim_fund df ON fp.amfi_code = df.amfi_code
WHERE
    fp.expense_ratio_pct IS NOT NULL
GROUP BY
    df.category
ORDER BY
    average_expense_ratio_percent DESC;
'''
display(pd.read_sql_query(query9, engine))



--- SQL Query 9: Average expense ratio by fund category ---


,category,average_expense_ratio_percent
0,ELSS,160.000000
1,Index,157.000000
2,Flexi Cap,154.500000
3,Large & Mid Cap,152.000000
4,Value,141.000000
5,Mid Cap,136.857143
6,Small Cap,135.166667
7,Large Cap,126.428571
8,Index/ETF,89.000000
9,Liquid,71.000000


In [72]:

print("\n--- SQL Query 10: Funds with Morningstar rating of 5 and positive 3-year return ---")
query10 = '''
SELECT
    df.scheme_name,
    df.fund_house,
    fp.morningstar_rating,
    fp.return_3yr_pct * 100 AS "3_Year_Return_Percent"
FROM
    fact_performance fp
JOIN
    dim_fund df ON fp.amfi_code = df.amfi_code
WHERE
    fp.morningstar_rating = 5
    AND fp.return_3yr_pct > 0
ORDER BY
    fp.return_3yr_pct DESC;
'''
display(pd.read_sql_query(query10, engine))


--- SQL Query 10: Funds with Morningstar rating of 5 and positive 3-year return ---


,scheme_name,fund_house,morningstar_rating,3_Year_Return_Percent
0,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,5,2339.0
1,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,5,2238.0
2,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,5,1658.0
3,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,5,1565.0
4,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,5,1561.0
5,UTI Flexi Cap Fund - Regular - Growth,UTI Mutual Fund,5,1534.0
6,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,5,1518.0
7,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,5,1484.0
8,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,5,1481.0
9,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,5,1456.0


## Data Dictionary

This data dictionary provides a comprehensive overview of the columns across the `nav_df`, `transactions_df`, and `scheme_df` DataFrames, including their data types, business definitions, and source files.

### `nav_df` (Source: `nav_history.csv`)

| Column Name | Data Type   | Business Definition                           | Source File         |
| :---------- | :---------- | :-------------------------------------------- | :------------------ |
| `amfi_code` | `int64`     | Unique identifier for a mutual fund scheme. | `nav_history.csv`   |
| `date`      | `datetime64`| Date of the Net Asset Value (NAV).            | `nav_history.csv`   |
| `nav`       | `float64`   | Net Asset Value per unit of the scheme.     | `nav_history.csv`   |

### `transactions_df` (Source: `investor_transactions.csv`)

| Column Name          | Data Type    | Business Definition                                       | Source File                   |
| :------------------- | :----------- | :-------------------------------------------------------- | :---------------------------- |
| `investor_id`        | `object`     | Unique identifier for an investor.                        | `investor_transactions.csv`   |
| `transaction_date`   | `datetime64` | Date when the transaction occurred.                       | `investor_transactions.csv`   |
| `amfi_code`          | `int64`      | Unique identifier for the mutual fund scheme involved.    | `investor_transactions.csv`   |
| `transaction_type`   | `object`     | Type of transaction (SIP, Lumpsum, Redemption).           | `investor_transactions.csv`   |
| `amount_inr`         | `int64`      | Amount of the transaction in Indian Rupees.               | `investor_transactions.csv`   |
| `state`              | `object`     | State where the investor resides.                         | `investor_transactions.csv`   |
| `city`               | `object`     | City where the investor resides.                          | `investor_transactions.csv`   |
| `city_tier`          | `object`     | Tier classification of the city (e.g., Tier 1, Tier 2). | `investor_transactions.csv`   |
| `age_group`          | `object`     | Age range of the investor.                                | `investor_transactions.csv`   |
| `gender`             | `object`     | Gender of the investor.                                   | `investor_transactions.csv`   |
| `annual_income_lakh` | `float64`    | Annual income of the investor in Lakhs.                   | `investor_transactions.csv`   |
| `payment_mode`       | `object`     | Mode of payment for the transaction.                      | `investor_transactions.csv`   |
| `kyc_status`         | `object`     | Know Your Customer (KYC) verification status of the investor. | `investor_transactions.csv`   |

### `scheme_df` (Source: `scheme_performance.csv`)

| Column Name          | Data Type    | Business Definition                                       | Source File                  |
| :------------------- | :----------- | :-------------------------------------------------------- | :--------------------------- |
| `amfi_code`          | `int64`      | Unique identifier for a mutual fund scheme.               | `scheme_performance.csv`     |
| `scheme_name`        | `object`     | Name of the mutual fund scheme.                           | `scheme_performance.csv`     |
| `fund_house`         | `object`     | Name of the asset management company (AMC) or fund house. | `scheme_performance.csv`     |
| `category`           | `object`     | Category of the mutual fund (e.g., Equity, Debt, Hybrid). | `scheme_performance.csv`     |
| `plan`               | `object`     | Plan type of the scheme (e.g., Regular, Direct).          | `scheme_performance.csv`     |
| `return_1yr_pct`     | `float64`    | 1-year return percentage of the scheme.                   | `scheme_performance.csv`     |
| `return_3yr_pct`     | `float64`    | 3-year return percentage of the scheme.                   | `scheme_performance.csv`     |
| `return_5yr_pct`     | `float64`    | 5-year return percentage of the scheme.                   | `scheme_performance.csv`     |
| `benchmark_3yr_pct`  | `float64`    | 3-year return percentage of the benchmark index.          | `scheme_performance.csv`     |
| `alpha`              | `float64`    | Measures fund performance relative to its benchmark.      | `scheme_performance.csv`     |
| `beta`               | `float64`    | Measures the volatility of a fund relative to the market. | `scheme_performance.csv`     |
| `sharpe_ratio`       | `float64`    | Measures risk-adjusted return.                            | `scheme_performance.csv`     |
| `sortino_ratio`      | `float64`    | Measures risk-adjusted return, focusing on downside risk. | `scheme_performance.csv`     |
| `std_dev_ann_pct`    | `float64`    | Annualized standard deviation of returns (volatility).    | `scheme_performance.csv`     |
| `max_drawdown_pct`   | `float64`    | Maximum decline from a peak to a trough.                  | `scheme_performance.csv`     |
| `aum_crore`          | `int64`      | Assets Under Management of the scheme in Crores.          | `scheme_performance.csv`     |
| `expense_ratio_pct`  | `float64`    | Expense ratio of the scheme as a percentage.              | `scheme_performance.csv`     |
| `morningstar_rating` | `int64`      | Morningstar rating of the scheme (1 to 5 stars).          | `scheme_performance.csv`     |
| `risk_grade`         | `object`     | Risk grading of the scheme.                               | `scheme_performance.csv`     |